In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Singularity Machine Learning - Classification: Uma Qiskit Function da Multiverse Computing
*Veja a [referência de API](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity)*


<Accordion>
<AccordionItem title="Versões dos pacotes">

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
scikit-learn~=1.8.0
```
</AccordionItem>
</Accordion>
> **Note:** * As Qiskit Functions são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium, Flex e On-Prem (via API da IBM Quantum Platform). Elas estão em status de pré-lançamento e sujeitas a alterações.

## Visão geral
Com a função "Singularity Machine Learning - Classification", você pode resolver problemas reais de aprendizado de máquina em hardware quântico sem precisar de expertise em computação quântica. Essa função de aplicação, baseada em métodos de ensemble, é um classificador híbrido. Ela utiliza métodos clássicos como boosting, bagging e stacking para o treinamento inicial do ensemble. Em seguida, algoritmos quânticos como o variational quantum eigensolver (VQE) e o quantum approximate optimization algorithm (QAOA) são empregados para aprimorar a diversidade, as capacidades de generalização e a complexidade geral do ensemble treinado.

Ao contrário de outras soluções de aprendizado de máquina quântico, essa função é capaz de lidar com conjuntos de dados em larga escala, com milhões de exemplos e features, sem ser limitada pelo número de qubits no QPU alvo. O número de qubits apenas determina o tamanho do ensemble que pode ser treinado. Ela também é altamente flexível e pode ser usada para resolver problemas de classificação em uma ampla gama de domínios, incluindo finanças, saúde e segurança cibernética.
Ela consegue consistentemente alta precisão em problemas classicamente desafiadores que envolvem conjuntos de dados de alta dimensionalidade, ruidosos e desbalanceados.
![Como funciona](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Ela foi construída para:
1. Engenheiros e cientistas de dados em empresas que buscam aprimorar suas ofertas tecnológicas integrando aprendizado de máquina quântico em seus produtos e serviços,
2. Pesquisadores em laboratórios de pesquisa quântica que exploram aplicações de aprendizado de máquina quântico e querem aproveitar a computação quântica para tarefas de classificação, e
3. Estudantes e professores em instituições educacionais em cursos como aprendizado de máquina, que buscam demonstrar as vantagens da computação quântica.

O exemplo a seguir demonstra suas diversas funcionalidades, incluindo `create`, `list`, `fit` e `predict`, e demonstra seu uso em um problema sintético composto por dois semicírculos entrelaçados, um problema notoriamente desafiador devido à sua fronteira de decisão não linear.


## Descrição da função
Essa Qiskit Function permite que os usuários resolvam problemas de classificação binária usando o classificador de ensemble aprimorado quanticamente da Singularity. Por trás dos panos, ela usa uma abordagem híbrida para treinar classicamente um ensemble de classificadores no conjunto de dados rotulado e, em seguida, otimizá-lo para máxima diversidade e generalização usando o Quantum Approximate Optimization Algorithm (QAOA) nos QPUs da IBM&reg;. Por meio de uma interface amigável, os usuários podem configurar um classificador de acordo com seus requisitos, treiná-lo no conjunto de dados de sua escolha e usá-lo para fazer previsões em um conjunto de dados nunca visto anteriormente.

Para resolver um problema genérico de classificação:
1. Pré-processe o conjunto de dados e divida-o em conjuntos de treinamento e teste. Opcionalmente, você pode dividir ainda mais o conjunto de treinamento em conjuntos de treinamento e validação. Isso pode ser feito usando [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Se o conjunto de treinamento estiver desbalanceado, você pode reamostrar para balancear as classes usando [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Faça upload dos conjuntos de treinamento, validação e teste separadamente para o armazenamento da função usando o método `file_upload` do catálogo, passando o caminho relevante a cada vez.
4. Inicialize o classificador quântico usando a ação `create` da função, que aceita hiperparâmetros como o número e os tipos de aprendizes, a regularização (valor lambda) e opções de otimização incluindo o número de camadas, o tipo de otimizador clássico, o backend quântico, entre outros.
5. Treine o classificador quântico no conjunto de treinamento usando a ação `fit` da função, passando o conjunto de treinamento rotulado e o conjunto de validação, se aplicável.
6. Faça previsões no conjunto de teste nunca visto anteriormente usando a ação `predict` da função.
---
## Primeiros passos
Autentique-se usando sua [chave de API da IBM Quantum Platform](http://quantum.cloud.ibm.com/) e selecione a Qiskit Function da seguinte forma:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## Exemplos
### Classificar um conjunto de dados
Neste exemplo, você vai usar a função "Singularity Machine Learning - Classification" para classificar um conjunto de dados composto por dois semicírculos entrelaçados em formato de lua. O conjunto de dados é sintético, bidimensional e rotulado com rótulos binários. Ele foi criado para ser desafiador para algoritmos como clustering baseado em centróide e classificação linear.
![Conjunto de dados moons](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
Ao longo deste processo, você vai aprender como criar o classificador, ajustá-lo aos dados de treinamento, usá-lo para prever nos dados de teste e excluir o classificador quando terminar.
Antes de começar, você precisa instalar o [scikit-learn](https://scikit-learn.org/). Instale-o usando o seguinte comando:

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


Execute os seguintes passos:

1) Crie o conjunto de dados sintético usando a função [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) do [scikit-learn](https://scikit-learn.org/).
2) Faça upload do conjunto de dados sintético gerado para o [diretório de dados compartilhado](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Crie o classificador aprimorado quanticamente usando a ação [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create).
4) Liste seus classificadores usando a ação [`list`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#list).
5) Treine o classificador nos dados de treinamento usando a ação [`fit`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#fit).
6) Use o classificador treinado para prever nos dados de teste usando a ação [`predict`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#predict).
7) Exclua o classificador usando a ação [`delete`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#delete).
8) Limpe ao terminar.
**Passo 1.** Importe os módulos necessários e gere o conjunto de dados sintético, depois divida-o em conjuntos de treinamento e teste.

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Step 3.** Create a quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Step 4.** Train the quantum-enhanced classifier using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


**Passo 3.** Crie um classificador aprimorado quanticamente usando a ação [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create).

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


**Step 6.** Delete the quantum-enhanced classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

### `create_fit_predict` example

The following example demonstrates the [`create_fit_predict`](/docs/api/functions/multiverse-computing-singularity#create_fit_predict) action.

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Visit the [API reference](/docs/api/functions/multiverse-computing-singularity) for this Qiskit Function.
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117).

</Admonition>